In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_PATH = r"D:\RAC-AI Projects\Khmer-Text-Full-System-Rith\NLP-python_service\models\khmer-mT5-news-summarization"

print("Loading model...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)

# Load model
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH, dtype=torch.float16 if device == "cuda" else torch.float32
)

model.to(device)
model.eval()

print("Model loaded successfully!")

Loading model...
Device: cuda


W0402 22:16:54.977000 33864 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Model loaded successfully!


In [2]:
def summarize(text: str, num_summaries: int = 1):
    text = text.strip()
    if not text:
        return {"summaries": [""]}

    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=200,  # increase max tokens
                min_length=80,  # enforce longer minimum
                do_sample=True,
                top_k=50,
                top_p=0.95,
                temperature=0.7,
                num_return_sequences=num_summaries,  # dynamic
                no_repeat_ngram_size=3,
                early_stopping=False,  # allow model to reach longer outputs
            )

        summaries = []
        for out in outputs:
            summary = tokenizer.decode(out, skip_special_tokens=True).strip()
            if summary and not summary.endswith("។"):
                summary += "។"
            summaries.append(summary)

        if num_summaries == 1:
            return {"summary": summaries[0]}
        return {"summaries": summaries}

    except Exception as e:
        return {"error": str(e)}

In [3]:
text_to_summarize = """
ប្រធានាធិបតីអាមេរិក លោក ដូណាល់ ត្រាំ បានគិតគូរជាយូរណាស់មកហើយចង់ឱ្យសហរដ្ឋអាមេរិកគ្រប់គ្រងដែនដី Greenland ជាគំនិតមួយដែលធ្វើឱ្យបណ្ដាមេដឹកនាំអឺរ៉ុប និងប្រជាជន Greenland ខឹងសម្បារផង និងសើចចំអកផង។ ប៉ុន្តែនៅបន្ទាប់ពីសហរដ្ឋអាមេរិកវាយប្រហារលើវ៉េណេស៊ុយអេឡា និងចាប់ខ្លួនលោក នីកូឡាស់ ម៉ាឌូរ៉ូ កាលពីថ្ងៃទី៣ ខែមករា វាបានបង្ហាញថាមហិច្ឆតារបស់លោក ត្រាំ ចង់គ្រប់គ្រងដែនដីក្នុងតំបន់អាក់ទិកមួយនេះហាក់មានភាពប្រាកដប្រជាជាងពេលណាៗទាំងអស់។
Greenland ដែលជាតំបន់ស្វយ័តមួយរបស់ដាណឺម៉ាក និងមានប្រជាជនរស់នៅ ៥៧០០០នាក់ បានទទួលឱ្យសហរដ្ឋអាមេរិកដាក់មូលដ្ឋានទ័ពមួយរួចទៅហើយ។ តែរដ្ឋបាលលោក ត្រាំចង់បាន សិទិ្ធអំណាចគ្រប់គ្រងបន្ថែមលើដែនដីមួយនេះ ដោយសារតែ Greenland មានទីភូមិសាស្ត្រយុទ្ធសាស្ត្រយ៉ាងសំខាន់នៅក្នុងតំបន់អាក់ទិកដើម្បីការពារផលប្រយោជន៍សន្តិសុខសហរដ្ឋអាមេរិក និង NATO។ «យើងត្រូវការ Greenland សម្រាប់ការការពារ»។ នេះជាអ្វីដែលលោក ត្រាំ បានប្រាប់កាសែតអាមេរិក The Atlantic កាលពីថ្ងៃទី៤ ខែមករា ជាមួយការអះអាងថា Greenland កំពុងត្រូវបានហ៊ុមព័ទ្ធដោយកប៉ាល់ចិន និងរុស្ស៉ី។ Greenland ក៏សម្បូរផងដែរធនធានធម្មជាតិ ក្នុងនោះរួមមានឧស្ម័នធម្មជាតិ ប្រេង និងរ៉ែដែលត្រូវបានប្រើក្នុងបច្ចេកវិទ្យា និងយោធាដែលកំពុង ក្លាយជាចំណុចក្ដៅមួយក្នុងសង្រ្គាមពាណិជ្ជកម្មរវាងសហរដ្ឋអាមេរិក និងចិន។
តែទោះជា
"""
data = summarize(text_to_summarize, num_summaries=1)
print("Summary:\n", data["summary"])

Summary:
 លោក ដូណាល់ ត្រាំ បាន គិតគូរ ថា អាមេរិក ចង់ឱ្យ Greenland គ្រប់គ្រងដែនដី របស់ Greenland ជាគំនិត មួយ ដែលធ្វើឱ្យបណ្ដាមេដឹកនាំអឺរ៉ុប និង ប្រជាជន Greenland ខឹងសម្បារ ។ ការបង្ហាញ ថា Greenland កំពុងត្រូវបានហ៊ុមព័ទ្ធដោយកប៉ាល់ចិន និងរុស្ស៉ី ។ Greenland ក៏សម្បូរធនធានធម្មជាតិ និងឧស្ម័នធម្មជាតិ ប្រេង និងរ៉ែ ដែល ត្រូវបានប្រើ ក្នុងបច្ចេកវិទ្យា និងយោធា ។


In [ ]:
# import requests

# # Replace with your HF Space URL
# API_URL = "https://angkor96-khmer-mT5-news-summarization-api.hf.space/summarize"

# text_to_summarize = """
# អង់គ្លេស៖ ក្លិប Liverpool នឹងចូលប្រកួតប្រជែងជាមួយក្លិប Manchester United ក្នុងការដាក់សំណើតាមទិញយកខ្សែបម្រើ Jack Grealish របស់ Aston Villa ។
# កីឡាករអង់គ្លេសរូបនេះ ពិតជាមានភាពលេចធ្លោ បើទោះបីជាគេមិនបានលេងឱ្យក្លិបធំៗទាំង៦នៅ Premier League ក៏ដោយ ស្របពេលកាលពីដើមរដូវកាល Grealish មានពាក្យចចាមអារ៉ាមថា បម្រុងនឹងផ្ទេរទៅចូលរួមជាមួយ Manchester United ទៅហើយ។ ពេលនេះ Liverpool កំពុងត្រូវការរុះរើ និងកសាងក្រុមឡើងវិញ បន្ទាប់ពីចាញ់ការប្រកួតក្នុងដី ដល់ទៅ៦ប្រកួតជាប់ៗគ្នា ហើយភាគីរបស់លោក Jurgen Klopp បានដាក់គោលដៅយកខ្សែបម្រើវ័យ២៥ឆ្នាំរូបនេះ មករួមក្រុម ខណៈខ្សែប្រយុទ្ធដូចជា Mohamed Salah, Sadio Mane និង Roberto Firmino ហាក់បីដូចជាធ្លាក់ទម្រង់លេងជាងរដូវកាលមុន។ Liverpool អាចទិញយក Jack Grealish បានតែត្រូវចាយប្រមាណ ៩០លានផោននៅរដូវក្ដៅខាងមុខ ប៉ុន្តែ Man UTD នៅតែជាក្លិបដែលមានប្រៀបជាងក្នុងការចរចាយក Grealish ៕
# """

# # If public, just call without headers
# response = requests.post(API_URL, json={"text": text_to_summarize})

# if response.status_code == 200:
#     data = response.json()
#     print("Summary:\n", data["summary"])
# else:
#     print("Error:", response.status_code, response.text)

Summary:
 លោក Jurgen Klopp បានដាក់សំណើតាមទិញយក Jack Grealish របស់ Aston Villa ដើម្បីចរចាយក Jack Giorgilish ។ គាត់ កំពុងត្រូវការរុះរើ និងកសាង ក្រុមឡើងវិញ បន្ទាប់ពី ចាញ់ការប្រកួតក្នុងដី ដល់ ៧ ប្រកួត ។ Liverpool នឹងចូលរួម ជាមួយ Manchester United មុនពេល ចាញ់ ប្រកួត ក្នុងដី ដល់ ៦ ប្រកួត បន្ទាប់ពីចាញ់ពាន Premier League ។ ក្លិប Liverpool បានដាក់គោលដៅទិញ Jack Greallish រហូតដល់ ៩០លានផោន នៅរដូវក្ដៅខាងមុខ ប៉ុន្តែ Man UTD នៅតែជាក្លិប ដែល នៅតែប្រៀបជាង ក្នុង ការចរចាយក Grealisem ។
